In [2]:
import pygame
from pygame.locals import *
from OpenGL.GL import *
from OpenGL.GLU import *
import math

# Initialize Pygame and OpenGL
pygame.init()
display = (800, 600)
pygame.display.set_mode(display, DOUBLEBUF | OPENGL)
pygame.display.set_caption("3D Neon Heart")

# Generate the 3D Heart Vertices using a parametric heart equation
def generate_heart_mesh():
    vertices = []
    # Loop through angles to create a smooth 3D shell
    for u in [i * 0.1 for i in range(-32, 32)]:
        for v in [i * 0.05 for i in range(-31, 31)]:
            # 3D Heart Formula
            x = 16 * math.sin(u) ** 3
            y = 13 * math.cos(u) - 5 * math.cos(2*u) - 2 * math.cos(3*u) - math.cos(4*u)
            z = v * 6  # Gives it 3D depth
            
            # Apply a narrowing factor to the back and front to close the 3D shape
            scale_depth = 1.0 - (abs(v) / 1.5)
            if scale_depth < 0: scale_depth = 0
            x *= scale_depth
            y *= scale_depth
            
            vertices.append((x * 0.1, y * 0.1, z * 0.1))
    return vertices

vertices = generate_heart_mesh()

# Function to create a text texture
def create_text_texture(text, font_size=32):
    font = pygame.font.SysFont("Arial", font_size, bold=True)
    # Render text in a bright cyan/neon color for contrast
    text_surface = font.render(text, True, (0, 255, 255), (0, 0, 0)) 
    text_data = pygame.image.tostring(text_surface, "RGBA", True)
    width, height = text_surface.get_size()

    texture_id = glGenTextures(1)
    glBindTexture(GL_TEXTURE_2D, texture_id)
    glTexImage2D(GL_TEXTURE_2D, 0, GL_RGBA, width, height, 0, GL_RGBA, GL_UNSIGNED_BYTE, text_data)
    glTexParameteri(GL_TEXTURE_2D, GL_TEXTURE_MIN_FILTER, GL_LINEAR)
    glTexParameteri(GL_TEXTURE_2D, GL_TEXTURE_MAG_FILTER, GL_LINEAR)
    
    return texture_id, width, height

text_id, txt_w, txt_h = create_text_texture("Haseeb Love Ahmad Mughal", 28)

# OpenGL Setup
gluPerspective(45, (display[0] / display[1]), 0.1, 50.0)
glTranslatef(0.0, 0.0, -20)
glEnable(GL_BLEND)
glBlendFunc(GL_SRC_ALPHA, GL_ONE_MINUS_SRC_ALPHA)
glEnable(GL_DEPTH_TEST)

# Animation variables
clock = pygame.time.Clock()
angle_y = 0
pulse_time = 0

# Main Loop
running = True
while running:
    for event in pygame.event.get():
        if event.type == pygame.QUIT:
            running = False

    # Dynamic pulsing logic using a sine wave
    pulse_time += 0.05
    pulse_scale = 1.0 + 0.12 * math.sin(pulse_time)
    angle_y += 1  # Continuous rotation

    # Clear screen with a dark neon-blue/violet ambient background color
    glClearColor(0.02, 0.0, 0.08, 1.0)
    glClear(GL_COLOR_BUFFER_BIT | GL_DEPTH_BUFFER_BIT)

    glPushMatrix()
    glRotatef(angle_y, 0, 1, 0)      # Rotate around Y axis
    glScalef(pulse_scale, pulse_scale, pulse_scale) # Apply pulsing animation
    
    # 1. DRAW BACKGROUND SHADOW (Slightly larger, dark blurred heart effect)
    glDisable(GL_DEPTH_TEST) # Draw background effects first safely
    glColor4f(0.2, 0.0, 0.0, 0.3) # Dark, semi-transparent red
    glPointSize(8.0)
    glBegin(GL_POINTS)
    for v in vertices:
        glVertex3f(v[0] * 1.1, v[1] * 1.1, v[2] * 1.1)
    glEnd()

    # 2. DRAW NEON GLOW (Outer Soft Layer)
    glColor4f(1.0, 0.0, 0.3, 0.2) # Soft neon pink/red glow
    glPointSize(6.0)
    glBegin(GL_POINTS)
    for v in vertices:
        glVertex3f(v[0], v[1], v[2])
    glEnd()

    # 3. DRAW INNER CORE (Bright Red/White Core Layer)
    glEnable(GL_DEPTH_TEST)
    glColor4f(1.0, 0.1, 0.1, 0.9) # Bright intense red core
    glPointSize(3.0)
    glBegin(GL_POINTS)
    for v in vertices:
        glVertex3f(v[0], v[1], v[2])
    glEnd()

    # 4. DRAW THE TEXT IN THE MIDDLE
    glEnable(GL_TEXTURE_2D)
    glBindTexture(GL_TEXTURE_2D, text_id)
    glColor4f(1.0, 1.0, 1.0, 1.0) # Keep texture colors true
    
    # Scale text down to fit perfectly inside the heart coordinates
    tw = txt_w * 0.015
    th = txt_h * 0.015
    
    glBegin(GL_QUADS)
    glTexCoord2f(0, 0); glVertex3f(-tw/2, -th/2, 0.1)  # Shifted slightly forward on Z
    glTexCoord2f(1, 0); glVertex3f(tw/2, -th/2, 0.1)
    glTexCoord2f(1, 1); glVertex3f(tw/2, th/2, 0.1)
    glTexCoord2f(0, 1); glVertex3f(-tw/2, th/2, 0.1)
    glEnd()
    glDisable(GL_TEXTURE_2D)

    glPopMatrix()

    pygame.display.flip()
    clock.tick(60)

pygame.quit()

In [3]:
import pygame
from pygame.locals import *
from OpenGL.GL import *
from OpenGL.GLU import *
import math
import random

# Initialize Pygame and OpenGL
pygame.init()
display = (1000, 750) # Made the window slightly larger for the bigger heart
pygame.display.set_mode(display, DOUBLEBUF | OPENGL)
pygame.display.set_caption("3D Neon Sprinkle Heart")

# Generate a larger, scattered "sprinkle" 3D Heart
def generate_sprinkle_heart():
    vertices = []
    # Increase range to generate more points for a dense sprinkle field
    for u in [i * 0.05 for i in range(-64, 64)]:
        for v in [i * 0.05 for i in range(-31, 31)]:
            # Base 3D Heart Formula
            base_x = 16 * math.sin(u) ** 3
            base_y = 13 * math.cos(u) - 5 * math.cos(2*u) - 2 * math.cos(3*u) - math.cos(4*u)
            base_z = v * 8  
            
            # Apply narrowing factor for 3D depth
            scale_depth = 1.0 - (abs(v) / 1.5)
            if scale_depth < 0: scale_depth = 0
            base_x *= scale_depth
            base_y *= scale_depth
            
            # SCATTER EFFECT: Add random offsets to make them look like floating sprinkles
            # Increase the multiplier to make the sprinkles tighter or more exploded
            sprinkle_x = (base_x + random.uniform(-2.5, 2.5)) * 0.18 # Increased multiplier for bigger size
            sprinkle_y = (base_y + random.uniform(-2.5, 2.5)) * 0.18
            sprinkle_z = (base_z + random.uniform(-2.5, 2.5)) * 0.18
            
            vertices.append((sprinkle_x, sprinkle_y, sprinkle_z))
    return vertices

vertices = generate_sprinkle_heart()

# Function to create a text texture
def create_text_texture(text, font_size=32):
    font = pygame.font.SysFont("Arial", font_size, bold=True)
    text_surface = font.render(text, True, (0, 255, 255), (0, 0, 0)) 
    text_data = pygame.image.tostring(text_surface, "RGBA", True)
    width, height = text_surface.get_size()

    texture_id = glGenTextures(1)
    glBindTexture(GL_TEXTURE_2D, texture_id)
    glTexImage2D(GL_TEXTURE_2D, 0, GL_RGBA, width, height, 0, GL_RGBA, GL_UNSIGNED_BYTE, text_data)
    glTexParameteri(GL_TEXTURE_2D, GL_TEXTURE_MIN_FILTER, GL_LINEAR)
    glTexParameteri(GL_TEXTURE_2D, GL_TEXTURE_MAG_FILTER, GL_LINEAR)
    
    return texture_id, width, height

text_id, txt_w, txt_h = create_text_texture("Haseeb Love Ahmad Mughal", 36) # Bumped up font size

# OpenGL Setup
gluPerspective(45, (display[0] / display[1]), 0.1, 50.0)
glTranslatef(0.0, 0.0, -22) # Adjusted camera depth to fit the massive heart
glEnable(GL_BLEND)
glBlendFunc(GL_SRC_ALPHA, GL_ONE_MINUS_SRC_ALPHA)
glEnable(GL_DEPTH_TEST)

# Animation variables
clock = pygame.time.Clock()
angle_y = 0
pulse_time = 0

# Main Loop
running = True
while running:
    for event in pygame.event.get():
        if event.type == pygame.QUIT:
            running = False

    # Pulsing logic
    pulse_time += 0.04
    pulse_scale = 1.0 + 0.15 * math.sin(pulse_time) # Slightly deeper pulse animation
    angle_y += 0.8  # Smoother rotation speed

    # Dark neon ambient background
    glClearColor(0.02, 0.0, 0.08, 1.0)
    glClear(GL_COLOR_BUFFER_BIT | GL_DEPTH_BUFFER_BIT)

    glPushMatrix()
    glRotatef(angle_y, 0, 1, 0)      
    glScalef(pulse_scale, pulse_scale, pulse_scale) 
    
    # 1. BACKGROUND SHADOW (Thick, dark neon sprinkle shadow)
    glDisable(GL_DEPTH_TEST)
    glColor4f(0.15, 0.0, 0.05, 0.25) 
    glPointSize(10.0) # Larger size for the background glow sprinkles
    glBegin(GL_POINTS)
    for v in vertices:
        glVertex3f(v[0] * 1.05, v[1] * 1.05, v[2] * 1.05)
    glEnd()

    # 2. NEON GLOW LAYER (Mid-size soft pink/red sprinkles)
    glColor4f(1.0, 0.0, 0.4, 0.3) 
    glPointSize(6.0)
    glBegin(GL_POINTS)
    for v in vertices:
        glVertex3f(v[0], v[1], v[2])
    glEnd()

    # 3. INNER CORE LAYER (Sharp, bright neon red core sprinkles)
    glEnable(GL_DEPTH_TEST)
    glColor4f(1.0, 0.2, 0.2, 0.95) 
    glPointSize(3.5)
    glBegin(GL_POINTS)
    for v in vertices:
        glVertex3f(v[0], v[1], v[2])
    glEnd()

    # 4. CENTERED TEXT 
    glEnable(GL_TEXTURE_2D)
    glBindTexture(GL_TEXTURE_2D, text_id)
    glColor4f(1.0, 1.0, 1.0, 1.0) 
    
    # Text scaling calibrated to look proportioned inside the larger heart
    tw = txt_w * 0.022
    th = txt_h * 0.022
    
    glBegin(GL_QUADS)
    glTexCoord2f(0, 0); glVertex3f(-tw/2, -th/2, 0.2)  
    glTexCoord2f(1, 0); glVertex3f(tw/2, -th/2, 0.2)
    glTexCoord2f(1, 1); glVertex3f(tw/2, th/2, 0.2)
    glTexCoord2f(0, 1); glVertex3f(-tw/2, th/2, 0.2)
    glEnd()
    glDisable(GL_TEXTURE_2D)

    glPopMatrix()

    pygame.display.flip()
    clock.tick(60)

pygame.quit()

In [4]:
import pygame
from pygame.locals import *
from OpenGL.GL import *
from OpenGL.GLU import *
import math
import random

# Initialize Pygame and OpenGL
pygame.init()
display = (1000, 750) 
pygame.display.set_mode(display, DOUBLEBUF | OPENGL)
pygame.display.set_caption("3D Neon Expanded Sprinkle Heart")

# Generate an expanded 3D Heart with more space between sprinkles
def generate_expanded_sprinkles():
    vertices = []
    # Using a slightly denser loop step to make sure the expanded shape stays defined
    for u in [i * 0.04 for i in range(-80, 80)]:
        for v in [i * 0.04 for i in range(-40, 40)]:
            # Base 3D Heart Formula
            base_x = 16 * math.sin(u) ** 3
            base_y = 13 * math.cos(u) - 5 * math.cos(2*u) - 2 * math.cos(3*u) - math.cos(4*u)
            base_z = v * 8  
            
            # Apply narrowing factor for 3D depth
            scale_depth = 1.0 - (abs(v) / 1.5)
            if scale_depth < 0: scale_depth = 0
            base_x *= scale_depth
            base_y *= scale_depth
            
            # EXPANSION & SPACING EFFECT:
            # We increased the random scatter range to create massive space between sprinkles,
            # and increased the scale multiplier (0.24) to expand the heart dramatically.
            sprinkle_x = (base_x + random.uniform(-6.0, 6.0)) * 0.24 
            sprinkle_y = (base_y + random.uniform(-6.0, 6.0)) * 0.24
            sprinkle_z = (base_z + random.uniform(-6.0, 6.0)) * 0.24
            
            vertices.append((sprinkle_x, sprinkle_y, sprinkle_z))
    return vertices

vertices = generate_expanded_sprinkles()

# Function to create a text texture
def create_text_texture(text, font_size=32):
    font = pygame.font.SysFont("Arial", font_size, bold=True)
    text_surface = font.render(text, True, (0, 255, 255), (0, 0, 0)) 
    text_data = pygame.image.tostring(text_surface, "RGBA", True)
    width, height = text_surface.get_size()

    texture_id = glGenTextures(1)
    glBindTexture(GL_TEXTURE_2D, texture_id)
    glTexImage2D(GL_TEXTURE_2D, 0, GL_RGBA, width, height, 0, GL_RGBA, GL_UNSIGNED_BYTE, text_data)
    glTexParameteri(GL_TEXTURE_2D, GL_TEXTURE_MIN_FILTER, GL_LINEAR)
    glTexParameteri(GL_TEXTURE_2D, GL_TEXTURE_MAG_FILTER, GL_LINEAR)
    
    return texture_id, width, height

text_id, txt_w, txt_h = create_text_texture("Haseeb Love Ahmad Mughal", 36)

# OpenGL Setup
gluPerspective(45, (display[0] / display[1]), 0.1, 50.0)
glTranslatef(0.0, 0.0, -25) # Pushed the camera slightly further back to capture the expanded heart
glEnable(GL_BLEND)
glBlendFunc(GL_SRC_ALPHA, GL_ONE_MINUS_SRC_ALPHA)
glEnable(GL_DEPTH_TEST)

# Animation variables
clock = pygame.time.Clock()
angle_y = 0
pulse_time = 0

# Main Loop
running = True
while running:
    for event in pygame.event.get():
        if event.type == pygame.QUIT:
            running = False

    # Pulsing logic
    pulse_time += 0.04
    pulse_scale = 1.0 + 0.12 * math.sin(pulse_time) 
    angle_y += 0.6  # Slightly slower rotation looks cleaner with scattered elements

    # Dark neon ambient background
    glClearColor(0.02, 0.0, 0.08, 1.0)
    glClear(GL_COLOR_BUFFER_BIT | GL_DEPTH_BUFFER_BIT)

    glPushMatrix()
    glRotatef(angle_y, 0, 1, 0)      
    glScalef(pulse_scale, pulse_scale, pulse_scale) 
    
    # 1. BACKGROUND SHADOW (Small points, low opacity)
    glDisable(GL_DEPTH_TEST)
    glColor4f(0.15, 0.0, 0.05, 0.2) 
    glPointSize(4.0) # Shrunk down from 10.0
    glBegin(GL_POINTS)
    for v in vertices:
        glVertex3f(v[0] * 1.05, v[1] * 1.05, v[2] * 1.05)
    glEnd()

    # 2. NEON GLOW LAYER (Small, soft pink sprinkles)
    glColor4f(1.0, 0.0, 0.4, 0.25) 
    glPointSize(2.5) # Shrunk down from 6.0
    glBegin(GL_POINTS)
    for v in vertices:
        glVertex3f(v[0], v[1], v[2])
    glEnd()

    # 3. INNER CORE LAYER (Tiny, sharp bright neon red sprinkles)
    glEnable(GL_DEPTH_TEST)
    glColor4f(1.0, 0.2, 0.2, 0.9) 
    glPointSize(1.0) # Shrunk down from 3.5 to make them look like sharp stars
    glBegin(GL_POINTS)
    for v in vertices:
        glVertex3f(v[0], v[1], v[2])
    glEnd()

    # 4. CENTERED TEXT 
    glEnable(GL_TEXTURE_2D)
    glBindTexture(GL_TEXTURE_2D, text_id)
    glColor4f(1.0, 1.0, 1.0, 1.0) 
    
    # Text sizing remains prominent right at the core of the cloud
    tw = txt_w * 0.022
    th = txt_h * 0.022
    
    glBegin(GL_QUADS)
    glTexCoord2f(0, 0); glVertex3f(-tw/2, -th/2, 0.2)  
    glTexCoord2f(1, 0); glVertex3f(tw/2, -th/2, 0.2)
    glTexCoord2f(1, 1); glVertex3f(tw/2, th/2, 0.2)
    glTexCoord2f(0, 1); glVertex3f(-tw/2, th/2, 0.2)
    glEnd()
    glDisable(GL_TEXTURE_2D)

    glPopMatrix()

    pygame.display.flip()
    clock.tick(60)

pygame.quit()

In [5]:
import pygame
from pygame.locals import *
from OpenGL.GL import *
from OpenGL.GLU import *
import math
import random

# Initialize Pygame and OpenGL
pygame.init()
display = (1000, 750) 
pygame.display.set_mode(display, DOUBLEBUF | OPENGL)
pygame.display.set_caption("3D Neon Expanded Sprinkle Heart")

# Generate an expanded 3D Heart with more space between sprinkles
def generate_expanded_sprinkles():
    vertices = []
    for u in [i * 0.04 for i in range(-80, 80)]:
        for v in [i * 0.04 for i in range(-40, 40)]:
            # Base 3D Heart Formula
            base_x = 16 * math.sin(u) ** 3
            base_y = 13 * math.cos(u) - 5 * math.cos(2*u) - 2 * math.cos(3*u) - math.cos(4*u)
            base_z = v * 8  
            
            # Apply narrowing factor for 3D depth
            scale_depth = 1.0 - (abs(v) / 1.5)
            if scale_depth < 0: scale_depth = 0
            base_x *= scale_depth
            base_y *= scale_depth
            
            # Expansion and spacing layout
            sprinkle_x = (base_x + random.uniform(-6.0, 6.0)) * 0.24 
            sprinkle_y = (base_y + random.uniform(-6.0, 6.0)) * 0.24
            sprinkle_z = (base_z + random.uniform(-6.0, 6.0)) * 0.24
            
            vertices.append((sprinkle_x, sprinkle_y, sprinkle_z))
    return vertices

vertices = generate_expanded_sprinkles()

# Function to create a transparent text texture
def create_text_texture(text, font_size=32):
    font = pygame.font.SysFont("Arial", font_size, bold=True)
    
    # FIX: Removed the (0, 0, 0) background argument. 
    # This renders the font on a fully transparent background surface.
    text_surface = font.render(text, True, (0, 255, 255)) 
    
    text_data = pygame.image.tostring(text_surface, "RGBA", True)
    width, height = text_surface.get_size()

    texture_id = glGenTextures(1)
    glBindTexture(GL_TEXTURE_2D, texture_id)
    glTexImage2D(GL_TEXTURE_2D, 0, GL_RGBA, width, height, 0, GL_RGBA, GL_UNSIGNED_BYTE, text_data)
    glTexParameteri(GL_TEXTURE_2D, GL_TEXTURE_MIN_FILTER, GL_LINEAR)
    glTexParameteri(GL_TEXTURE_2D, GL_TEXTURE_MAG_FILTER, GL_LINEAR)
    
    return texture_id, width, height

text_id, txt_w, txt_h = create_text_texture("Haseeb Love Ahmad Mughal", 36)

# OpenGL Setup
gluPerspective(45, (display[0] / display[1]), 0.1, 50.0)
glTranslatef(0.0, 0.0, -25) 
glEnable(GL_BLEND)
glBlendFunc(GL_SRC_ALPHA, GL_ONE_MINUS_SRC_ALPHA)
glEnable(GL_DEPTH_TEST)

# Animation variables
clock = pygame.time.Clock()
angle_y = 0
pulse_time = 0

# Main Loop
running = True
while running:
    for event in pygame.event.get():
        if event.type == pygame.QUIT:
            running = False

    # Pulsing logic
    pulse_time += 0.04
    pulse_scale = 1.0 + 0.12 * math.sin(pulse_time) 
    angle_y += 0.6  

    # Dark neon ambient background
    glClearColor(0.02, 0.0, 0.08, 1.0)
    glClear(GL_COLOR_BUFFER_BIT | GL_DEPTH_BUFFER_BIT)

    glPushMatrix()
    glRotatef(angle_y, 0, 1, 0)      
    glScalef(pulse_scale, pulse_scale, pulse_scale) 
    
    # 1. BACKGROUND SHADOW 
    glDisable(GL_DEPTH_TEST)
    glColor4f(0.15, 0.0, 0.05, 0.2) 
    glPointSize(4.0) 
    glBegin(GL_POINTS)
    for v in vertices:
        glVertex3f(v[0] * 1.05, v[1] * 1.05, v[2] * 1.05)
    glEnd()

    # 2. NEON GLOW LAYER 
    glColor4f(1.0, 0.0, 0.4, 0.25) 
    glPointSize(2.5) 
    glBegin(GL_POINTS)
    for v in vertices:
        glVertex3f(v[0], v[1], v[2])
    glEnd()

    # 3. INNER CORE LAYER 
    glEnable(GL_DEPTH_TEST)
    glColor4f(1.0, 0.2, 0.2, 0.9) 
    glPointSize(1.0) 
    glBegin(GL_POINTS)
    for v in vertices:
        glVertex3f(v[0], v[1], v[2])
    glEnd()

    # 4. CENTERED TEXT (Now completely floating without a bounding box)
    glEnable(GL_TEXTURE_2D)
    glBindTexture(GL_TEXTURE_2D, text_id)
    glColor4f(1.0, 1.0, 1.0, 1.0) 
    
    tw = txt_w * 0.022
    th = txt_h * 0.022
    
    glBegin(GL_QUADS)
    glTexCoord2f(0, 0); glVertex3f(-tw/2, -th/2, 0.2)  
    glTexCoord2f(1, 0); glVertex3f(tw/2, -th/2, 0.2)
    glTexCoord2f(1, 1); glVertex3f(tw/2, th/2, 0.2)
    glTexCoord2f(0, 1); glVertex3f(-tw/2, th/2, 0.2)
    glEnd()
    glDisable(GL_TEXTURE_2D)

    glPopMatrix()

    pygame.display.flip()
    clock.tick(60)

pygame.quit()

In [6]:
import pygame
from pygame.locals import *
from OpenGL.GL import *
from OpenGL.GLU import *
import math
import random

# Initialize Pygame and OpenGL
pygame.init()
display = (1000, 750) 
pygame.display.set_mode(display, DOUBLEBUF | OPENGL)
pygame.display.set_caption("3D Neon Expanded Sprinkle Heart")

# Generate an expanded 3D Heart with more space between sprinkles
def generate_expanded_sprinkles():
    vertices = []
    for u in [i * 0.04 for i in range(-80, 80)]:
        for v in [i * 0.04 for i in range(-40, 40)]:
            # Base 3D Heart Formula
            base_x = 16 * math.sin(u) ** 3
            base_y = 13 * math.cos(u) - 5 * math.cos(2*u) - 2 * math.cos(3*u) - math.cos(4*u)
            base_z = v * 8  
            
            # Apply narrowing factor for 3D depth
            scale_depth = 1.0 - (abs(v) / 1.5)
            if scale_depth < 0: scale_depth = 0
            base_x *= scale_depth
            base_y *= scale_depth
            
            # Expansion and spacing layout
            sprinkle_x = (base_x + random.uniform(-6.0, 6.0)) * 0.24 
            sprinkle_y = (base_y + random.uniform(-6.0, 6.0)) * 0.24
            sprinkle_z = (base_z + random.uniform(-6.0, 6.0)) * 0.24
            
            vertices.append((sprinkle_x, sprinkle_y, sprinkle_z))
    return vertices

vertices = generate_expanded_sprinkles()

# Function to create a transparent text texture
def create_text_texture(text, font_size=48):
    # Using a standard clean font (Arial/Segoe UI Symbol handles the text heart beautifully)
    font = pygame.font.SysFont("Segoe UI Symbol", font_size, bold=True)
    
    # Render with a bright glowing neon cyan color, completely transparent background
    text_surface = font.render(text, True, (0, 255, 255)) 
    
    text_data = pygame.image.tostring(text_surface, "RGBA", True)
    width, height = text_surface.get_size()

    texture_id = glGenTextures(1)
    glBindTexture(GL_TEXTURE_2D, texture_id)
    glTexImage2D(GL_TEXTURE_2D, 0, GL_RGBA, width, height, 0, GL_RGBA, GL_UNSIGNED_BYTE, text_data)
    glTexParameteri(GL_TEXTURE_2D, GL_TEXTURE_MIN_FILTER, GL_LINEAR)
    glTexParameteri(GL_TEXTURE_2D, GL_TEXTURE_MAG_FILTER, GL_LINEAR)
    
    return texture_id, width, height

# Updated text initialization string
text_id, txt_w, txt_h = create_text_texture("H❤A.M", 54)

# OpenGL Setup
gluPerspective(45, (display[0] / display[1]), 0.1, 50.0)
glTranslatef(0.0, 0.0, -25) 
glEnable(GL_BLEND)
glBlendFunc(GL_SRC_ALPHA, GL_ONE_MINUS_SRC_ALPHA)
glEnable(GL_DEPTH_TEST)

# Animation variables
clock = pygame.time.Clock()
angle_y = 0
pulse_time = 0

# Main Loop
running = True
while running:
    for event in pygame.event.get():
        if event.type == pygame.QUIT:
            running = False

    # Pulsing logic
    pulse_time += 0.04
    pulse_scale = 1.0 + 0.12 * math.sin(pulse_time) 
    angle_y += 0.6  

    # Dark neon ambient background
    glClearColor(0.02, 0.0, 0.08, 1.0)
    glClear(GL_COLOR_BUFFER_BIT | GL_DEPTH_BUFFER_BIT)

    glPushMatrix()
    glRotatef(angle_y, 0, 1, 0)      
    glScalef(pulse_scale, pulse_scale, pulse_scale) 
    
    # 1. BACKGROUND SHADOW 
    glDisable(GL_DEPTH_TEST)
    glColor4f(0.15, 0.0, 0.05, 0.2) 
    glPointSize(4.0) 
    glBegin(GL_POINTS)
    for v in vertices:
        glVertex3f(v[0] * 1.05, v[1] * 1.05, v[2] * 1.05)
    glEnd()

    # 2. NEON GLOW LAYER 
    glColor4f(1.0, 0.0, 0.4, 0.25) 
    glPointSize(2.5) 
    glBegin(GL_POINTS)
    for v in vertices:
        glVertex3f(v[0], v[1], v[2])
    glEnd()

    # 3. INNER CORE LAYER 
    glEnable(GL_DEPTH_TEST)
    glColor4f(1.0, 0.2, 0.2, 0.9) 
    glPointSize(1.0) 
    glBegin(GL_POINTS)
    for v in vertices:
        glVertex3f(v[0], v[1], v[2])
    glEnd()

    # 4. CENTERED TEXT 
    glEnable(GL_TEXTURE_2D)
    glBindTexture(GL_TEXTURE_2D, text_id)
    glColor4f(1.0, 1.0, 1.0, 1.0) 
    
    # Slightly adjusted text size multipliers to account for shorter string length
    tw = txt_w * 0.025
    th = txt_h * 0.025
    
    glBegin(GL_QUADS)
    glTexCoord2f(0, 0); glVertex3f(-tw/2, -th/2, 0.2)  
    glTexCoord2f(1, 0); glVertex3f(tw/2, -th/2, 0.2)
    glTexCoord2f(1, 1); glVertex3f(tw/2, th/2, 0.2)
    glTexCoord2f(0, 1); glVertex3f(-tw/2, th/2, 0.2)
    glEnd()
    glDisable(GL_TEXTURE_2D)

    glPopMatrix()

    pygame.display.flip()
    clock.tick(60)

pygame.quit()

In [8]:
import pygame
from pygame.locals import *
from OpenGL.GL import *
from OpenGL.GLU import *
import math
import random

# Initialize Pygame and OpenGL
pygame.init()
display = (1000, 750) 
pygame.display.set_mode(display, DOUBLEBUF | OPENGL)
pygame.display.set_caption("3D Neon Expanded Sprinkle Heart")

# Generate an expanded 3D Heart with more space between sprinkles
def generate_expanded_sprinkles():
    vertices = []
    for u in [i * 0.04 for i in range(-80, 80)]:
        for v in [i * 0.04 for i in range(-40, 40)]:
            # Base 3D Heart Formula
            base_x = 16 * math.sin(u) ** 3
            base_y = 13 * math.cos(u) - 5 * math.cos(2*u) - 2 * math.cos(3*u) - math.cos(4*u)
            base_z = v * 8  
            
            # Apply narrowing factor for 3D depth
            scale_depth = 1.0 - (abs(v) / 1.5)
            if scale_depth < 0: scale_depth = 0
            base_x *= scale_depth
            base_y *= scale_depth
            
            # Expansion and spacing layout
            sprinkle_x = (base_x + random.uniform(-6.0, 6.0)) * 0.24 
            sprinkle_y = (base_y + random.uniform(-6.0, 6.0)) * 0.24
            sprinkle_z = (base_z + random.uniform(-6.0, 6.0)) * 0.24
            
            vertices.append((sprinkle_x, sprinkle_y, sprinkle_z))
    return vertices

vertices = generate_expanded_sprinkles()

# Function to create a transparent text texture
def create_text_texture(text, font_size=72):
    # CHANGED: Using "Gabriola" for a gorgeous, cursive, flowing love-style font.
    # Fallback to standard system fonts like Monotype Corsiva or standard serif if unavailable.
    font_options = ["gabriola", "monotypecorsiva", "georgia", "timesnewroman"]
    font = pygame.font.SysFont(font_options, font_size, bold=True, italic=True)
    
    # Render with a bright glowing neon cyan color, completely transparent background
    text_surface = font.render(text, True, (0, 255, 255)) 
    
    text_data = pygame.image.tostring(text_surface, "RGBA", True)
    width, height = text_surface.get_size()

    texture_id = glGenTextures(1)
    glBindTexture(GL_TEXTURE_2D, texture_id)
    glTexImage2D(GL_TEXTURE_2D, 0, GL_RGBA, width, height, 0, GL_RGBA, GL_UNSIGNED_BYTE, text_data)
    glTexParameteri(GL_TEXTURE_2D, GL_TEXTURE_MIN_FILTER, GL_LINEAR)
    glTexParameteri(GL_TEXTURE_2D, GL_TEXTURE_MAG_FILTER, GL_LINEAR)
    
    return texture_id, width, height

# Text initialization string
text_id, txt_w, txt_h = create_text_texture("H❤A.M", 72)

# OpenGL Setup
gluPerspective(45, (display[0] / display[1]), 0.1, 50.0)
glTranslatef(0.0, 0.0, -25) 
glEnable(GL_BLEND)
glBlendFunc(GL_SRC_ALPHA, GL_ONE_MINUS_SRC_ALPHA)
glEnable(GL_DEPTH_TEST)

# Animation variables
clock = pygame.time.Clock()
angle_y = 0
pulse_time = 0

# Main Loop
running = True
while running:
    for event in pygame.event.get():
        if event.type == pygame.QUIT:
            running = False

    # Pulsing logic
    pulse_time += 0.04
    pulse_scale = 1.0 + 0.12 * math.sin(pulse_time) 
    angle_y += 0.6  

    # Dark neon ambient background
    glClearColor(0.02, 0.0, 0.08, 1.0)
    glClear(GL_COLOR_BUFFER_BIT | GL_DEPTH_BUFFER_BIT)

    glPushMatrix()
    glRotatef(angle_y, 0, 1, 0)      
    glScalef(pulse_scale, pulse_scale, pulse_scale) 
    
    # 1. BACKGROUND SHADOW 
    glDisable(GL_DEPTH_TEST)
    glColor4f(0.15, 0.0, 0.05, 0.2) 
    glPointSize(4.0) 
    glBegin(GL_POINTS)
    for v in vertices:
        glVertex3f(v[0] * 1.05, v[1] * 1.05, v[2] * 1.05)
    glEnd()

    # 2. NEON GLOW LAYER 
    glColor4f(1.0, 0.0, 0.4, 0.25) 
    glPointSize(2.5) 
    glBegin(GL_POINTS)
    for v in vertices:
        glVertex3f(v[0], v[1], v[2])
    glEnd()

    # 3. INNER CORE LAYER 
    glEnable(GL_DEPTH_TEST)
    glColor4f(1.0, 0.2, 0.2, 0.9) 
    glPointSize(1.0) 
    glBegin(GL_POINTS)
    for v in vertices:
        glVertex3f(v[0], v[1], v[2])
    glEnd()

    # 4. CENTERED TEXT 
    glEnable(GL_TEXTURE_2D)
    glBindTexture(GL_TEXTURE_2D, text_id)
    glColor4f(1.0, 1.0, 1.0, 1.0) 
    
    # Scaling adjusted slightly to make the cursive details pop out nicely inside the core
    tw = txt_w * 0.024
    th = txt_h * 0.024
    
    glBegin(GL_QUADS)
    glTexCoord2f(0, 0); glVertex3f(-tw/2, -th/2, 0.2)  
    glTexCoord2f(1, 0); glVertex3f(tw/2, -th/2, 0.2)
    glTexCoord2f(1, 1); glVertex3f(tw/2, th/2, 0.2)
    glTexCoord2f(0, 1); glVertex3f(-tw/2, th/2, 0.2)
    glEnd()
    glDisable(GL_TEXTURE_2D)

    glPopMatrix()

    pygame.display.flip()
    clock.tick(60)

pygame.quit()

In [9]:
import pygame
from pygame.locals import *
from OpenGL.GL import *
from OpenGL.GLU import *
import math
import random

# Initialize Pygame and OpenGL
pygame.init()
display = (1000, 750) 
pygame.display.set_mode(display, DOUBLEBUF | OPENGL)
pygame.display.set_caption("3D Neon Expanded Sprinkle Heart")

# Generate an expanded 3D Heart with more space between sprinkles
def generate_expanded_sprinkles():
    vertices = []
    for u in [i * 0.04 for i in range(-80, 80)]:
        for v in [i * 0.04 for i in range(-40, 40)]:
            # Base 3D Heart Formula
            base_x = 16 * math.sin(u) ** 3
            base_y = 13 * math.cos(u) - 5 * math.cos(2*u) - 2 * math.cos(3*u) - math.cos(4*u)
            base_z = v * 8  
            
            # Apply narrowing factor for 3D depth
            scale_depth = 1.0 - (abs(v) / 1.5)
            if scale_depth < 0: scale_depth = 0
            base_x *= scale_depth
            base_y *= scale_depth
            
            # Expansion and spacing layout
            sprinkle_x = (base_x + random.uniform(-6.0, 6.0)) * 0.24 
            sprinkle_y = (base_y + random.uniform(-6.0, 6.0)) * 0.24
            sprinkle_z = (base_z + random.uniform(-6.0, 6.0)) * 0.24
            
            vertices.append((sprinkle_x, sprinkle_y, sprinkle_z))
    return vertices

vertices = generate_expanded_sprinkles()

# Function to create a transparent text texture
def create_text_texture(text, font_size=72):
    font_options = ["gabriola", "monotypecorsiva", "georgia", "timesnewroman"]
    font = pygame.font.SysFont(font_options, font_size, bold=True, italic=True)
    
    # CHANGED: Render text with a gorgeous Neon Pink/Magenta color (255, 20, 147)
    text_surface = font.render(text, True, (255, 20, 147)) 
    
    text_data = pygame.image.tostring(text_surface, "RGBA", True)
    width, height = text_surface.get_size()

    texture_id = glGenTextures(1)
    glBindTexture(GL_TEXTURE_2D, texture_id)
    glTexImage2D(GL_TEXTURE_2D, 0, GL_RGBA, width, height, 0, GL_RGBA, GL_UNSIGNED_BYTE, text_data)
    glTexParameteri(GL_TEXTURE_2D, GL_TEXTURE_MIN_FILTER, GL_LINEAR)
    glTexParameteri(GL_TEXTURE_2D, GL_TEXTURE_MAG_FILTER, GL_LINEAR)
    
    return texture_id, width, height

# Text initialization string
text_id, txt_w, txt_h = create_text_texture("H❤A.M", 72)

# OpenGL Setup
gluPerspective(45, (display[0] / display[1]), 0.1, 50.0)
glTranslatef(0.0, 0.0, -25) 
glEnable(GL_BLEND)
glBlendFunc(GL_SRC_ALPHA, GL_ONE_MINUS_SRC_ALPHA)
glEnable(GL_DEPTH_TEST)

# Animation variables
clock = pygame.time.Clock()
angle_y = 0
pulse_time = 0

# Main Loop
running = True
while running:
    for event in pygame.event.get():
        if event.type == pygame.QUIT:
            running = False

    # Pulsing logic
    pulse_time += 0.04
    pulse_scale = 1.0 + 0.12 * math.sin(pulse_time) 
    angle_y += 0.6  

    # Dark neon deep space background
    glClearColor(0.01, 0.0, 0.05, 1.0)
    glClear(GL_COLOR_BUFFER_BIT | GL_DEPTH_BUFFER_BIT)

    glPushMatrix()
    glRotatef(angle_y, 0, 1, 0)      
    glScalef(pulse_scale, pulse_scale, pulse_scale) 
    
    # 1. BACKGROUND SHADOW (Dark, transparent cyan/blue backdrop)
    glDisable(GL_DEPTH_TEST)
    glColor4f(0.0, 0.1, 0.15, 0.2) 
    glPointSize(4.0) 
    glBegin(GL_POINTS)
    for v in vertices:
        glVertex3f(v[0] * 1.05, v[1] * 1.05, v[2] * 1.05)
    glEnd()

    # 2. NEON GLOW LAYER (Soft Outer Cyan Sprinkles)
    glColor4f(0.0, 0.8, 1.0, 0.25) 
    glPointSize(2.5) 
    glBegin(GL_POINTS)
    for v in vertices:
        glVertex3f(v[0], v[1], v[2])
    glEnd()

    # 3. INNER CORE LAYER (Sharp Bright Aqua/Cyan Sprinkles)
    glEnable(GL_DEPTH_TEST)
    glColor4f(0.3, 1.0, 1.0, 0.9) 
    glPointSize(1.0) 
    glBegin(GL_POINTS)
    for v in vertices:
        glVertex3f(v[0], v[1], v[2])
    glEnd()

    # 4. CENTERED TEXT 
    glEnable(GL_TEXTURE_2D)
    glBindTexture(GL_TEXTURE_2D, text_id)
    glColor4f(1.0, 1.0, 1.0, 1.0) 
    
    tw = txt_w * 0.024
    th = txt_h * 0.024
    
    glBegin(GL_QUADS)
    glTexCoord2f(0, 0); glVertex3f(-tw/2, -th/2, 0.2)  
    glTexCoord2f(1, 0); glVertex3f(tw/2, -th/2, 0.2)
    glTexCoord2f(1, 1); glVertex3f(tw/2, th/2, 0.2)
    glTexCoord2f(0, 1); glVertex3f(-tw/2, th/2, 0.2)
    glEnd()
    glDisable(GL_TEXTURE_2D)

    glPopMatrix()

    pygame.display.flip()
    clock.tick(60)

pygame.quit()

In [10]:
import pygame
from pygame.locals import *
from OpenGL.GL import *
from OpenGL.GLU import *
import math
import random

# Initialize Pygame and OpenGL
pygame.init()
display = (1000, 750) 
pygame.display.set_mode(display, DOUBLEBUF | OPENGL)
pygame.display.set_caption("3D Neon Rainbow Sprinkle Heart")

# Helper class to handle random smooth color shifting
class ColorManager:
    def __init__(self):
        self.current_text_color = [random.random(), random.random(), random.random()]
        self.target_text_color = [random.random(), random.random(), random.random()]
        self.current_heart_color = [random.random(), random.random(), random.random()]
        self.target_heart_color = [random.random(), random.random(), random.random()]
        self.last_change_time = pygame.time.get_ticks()

    def update(self):
        now = pygame.time.get_ticks()
        # Change target colors randomly every 600 milliseconds
        if now - self.last_change_time > 600:
            # We enforce high color values so the colors always stay bright/neon
            self.target_text_color = [random.uniform(0.5, 1.0), random.uniform(0.5, 1.0), random.uniform(0.5, 1.0)]
            self.target_heart_color = [random.uniform(0.4, 1.0), random.uniform(0.4, 1.0), random.uniform(0.4, 1.0)]
            self.last_change_time = now

        # Linear interpolation (LERP) for smooth transition between random colors
        for i in range(3):
            self.current_text_color[i] += (self.target_text_color[i] - self.current_text_color[i]) * 0.1
            self.current_heart_color[i] += (self.target_heart_color[i] - self.current_heart_color[i]) * 0.1

color_manager = ColorManager()

# Generate an expanded 3D Heart with more space between sprinkles
def generate_expanded_sprinkles():
    vertices = []
    for u in [i * 0.04 for i in range(-80, 80)]:
        for v in [i * 0.04 for i in range(-40, 40)]:
            base_x = 16 * math.sin(u) ** 3
            base_y = 13 * math.cos(u) - 5 * math.cos(2*u) - 2 * math.cos(3*u) - math.cos(4*u)
            base_z = v * 8  
            
            scale_depth = 1.0 - (abs(v) / 1.5)
            if scale_depth < 0: scale_depth = 0
            base_x *= scale_depth
            base_y *= scale_depth
            
            sprinkle_x = (base_x + random.uniform(-6.0, 6.0)) * 0.24 
            sprinkle_y = (base_y + random.uniform(-6.0, 6.0)) * 0.24
            sprinkle_z = (base_z + random.uniform(-6.0, 6.0)) * 0.24
            
            vertices.append((sprinkle_x, sprinkle_y, sprinkle_z))
    return vertices

vertices = generate_expanded_sprinkles()

# Function to create a transparent text texture (Rendered white initially so OpenGL can tint it)
def create_text_texture(text, font_size=72):
    font_options = ["gabriola", "monotypecorsiva", "georgia", "timesnewroman"]
    font = pygame.font.SysFont(font_options, font_size, bold=True, italic=True)
    
    # We render the text base texture in PURE WHITE.
    # This lets us use glColor4f dynamically to change its color to anything randomly on the fly!
    text_surface = font.render(text, True, (255, 255, 255)) 
    
    text_data = pygame.image.tostring(text_surface, "RGBA", True)
    width, height = text_surface.get_size()

    texture_id = glGenTextures(1)
    glBindTexture(GL_TEXTURE_2D, texture_id)
    glTexImage2D(GL_TEXTURE_2D, 0, GL_RGBA, width, height, 0, GL_RGBA, GL_UNSIGNED_BYTE, text_data)
    glTexParameteri(GL_TEXTURE_2D, GL_TEXTURE_MIN_FILTER, GL_LINEAR)
    glTexParameteri(GL_TEXTURE_2D, GL_TEXTURE_MAG_FILTER, GL_LINEAR)
    
    return texture_id, width, height

text_id, txt_w, txt_h = create_text_texture("H❤A.M", 72)

# OpenGL Setup
gluPerspective(45, (display[0] / display[1]), 0.1, 50.0)
glTranslatef(0.0, 0.0, -25) 
glEnable(GL_BLEND)
glBlendFunc(GL_SRC_ALPHA, GL_ONE_MINUS_SRC_ALPHA)
glEnable(GL_DEPTH_TEST)

# Animation variables
clock = pygame.time.Clock()
angle_y = 0
pulse_time = 0

# Main Loop
running = True
while running:
    for event in pygame.event.get():
        if event.type == pygame.QUIT:
            running = False

    # Update dynamic random colors
    color_manager.update()
    hc = color_manager.current_heart_color
    tc = color_manager.current_text_color

    # Pulsing logic
    pulse_time += 0.04
    pulse_scale = 1.0 + 0.12 * math.sin(pulse_time) 
    angle_y += 0.6  

    # Dark neon deep space background
    glClearColor(0.01, 0.0, 0.04, 1.0)
    glClear(GL_COLOR_BUFFER_BIT | GL_DEPTH_BUFFER_BIT)

    glPushMatrix()
    glRotatef(angle_y, 0, 1, 0)      
    glScalef(pulse_scale, pulse_scale, pulse_scale) 
    
    # 1. BACKGROUND SHADOW (Dark tint of the random heart color)
    glDisable(GL_DEPTH_TEST)
    glColor4f(hc[0]*0.2, hc[1]*0.2, hc[2]*0.2, 0.2) 
    glPointSize(4.0) 
    glBegin(GL_POINTS)
    for v in vertices:
        glVertex3f(v[0] * 1.05, v[1] * 1.05, v[2] * 1.05)
    glEnd()

    # 2. NEON GLOW LAYER (Soft translucent version of the random heart color)
    glColor4f(hc[0], hc[1], hc[2], 0.25) 
    glPointSize(2.5) 
    glBegin(GL_POINTS)
    for v in vertices:
        glVertex3f(v[0], v[1], v[2])
    glEnd()

    # 3. INNER CORE LAYER (Solid sharp version of the random heart color)
    glEnable(GL_DEPTH_TEST)
    glColor4f(hc[0], hc[1], hc[2], 0.9) 
    glPointSize(1.0) 
    glBegin(GL_POINTS)
    for v in vertices:
        glVertex3f(v[0], v[1], v[2])
    glEnd()

    # 4. CENTERED TEXT (Using the random text color coordinates)
    glEnable(GL_TEXTURE_2D)
    glBindTexture(GL_TEXTURE_2D, text_id)
    glColor4f(tc[0], tc[1], tc[2], 1.0) # Dynamically tints the white text asset
    
    tw = txt_w * 0.024
    th = txt_h * 0.024
    
    glBegin(GL_QUADS)
    glTexCoord2f(0, 0); glVertex3f(-tw/2, -th/2, 0.2)  
    glTexCoord2f(1, 0); glVertex3f(tw/2, -th/2, 0.2)
    glTexCoord2f(1, 1); glVertex3f(tw/2, th/2, 0.2)
    glTexCoord2f(0, 1); glVertex3f(-tw/2, th/2, 0.2)
    glEnd()
    glDisable(GL_TEXTURE_2D)

    glPopMatrix()

    pygame.display.flip()
    clock.tick(60)

pygame.quit()

In [11]:
import pygame
from pygame.locals import *
from OpenGL.GL import *
from OpenGL.GLU import *
import math
import random

# Initialize Pygame and OpenGL
pygame.init()
display = (1000, 750) 
pygame.display.set_mode(display, DOUBLEBUF | OPENGL)
pygame.display.set_caption("3D Neon Rainbow Sprinkle Heart")

# Helper class to handle random smooth color shifting
class ColorManager:
    def __init__(self):
        self.current_text_color = [random.random(), random.random(), random.random()]
        self.target_text_color = [random.random(), random.random(), random.random()]
        self.current_heart_color = [random.random(), random.random(), random.random()]
        self.target_heart_color = [random.random(), random.random(), random.random()]
        self.last_change_time = pygame.time.get_ticks()

    def update(self):
        now = pygame.time.get_ticks()
        if now - self.last_change_time > 600:
            self.target_text_color = [random.uniform(0.5, 1.0), random.uniform(0.5, 1.0), random.uniform(0.5, 1.0)]
            self.target_heart_color = [random.uniform(0.4, 1.0), random.uniform(0.4, 1.0), random.uniform(0.4, 1.0)]
            self.last_change_time = now

        for i in range(3):
            self.current_text_color[i] += (self.target_text_color[i] - self.current_text_color[i]) * 0.1
            self.current_heart_color[i] += (self.target_heart_color[i] - self.current_heart_color[i]) * 0.1

color_manager = ColorManager()

# Generate an expanded 3D Heart with more space between sprinkles
def generate_expanded_sprinkles():
    vertices = []
    for u in [i * 0.04 for i in range(-80, 80)]:
        for v in [i * 0.04 for i in range(-40, 40)]:
            base_x = 16 * math.sin(u) ** 3
            base_y = 13 * math.cos(u) - 5 * math.cos(2*u) - 2 * math.cos(3*u) - math.cos(4*u)
            base_z = v * 8  
            
            scale_depth = 1.0 - (abs(v) / 1.5)
            if scale_depth < 0: scale_depth = 0
            base_x *= scale_depth
            base_y *= scale_depth
            
            sprinkle_x = (base_x + random.uniform(-6.0, 6.0)) * 0.24 
            sprinkle_y = (base_y + random.uniform(-6.0, 6.0)) * 0.24
            sprinkle_z = (base_z + random.uniform(-6.0, 6.0)) * 0.24
            
            vertices.append((sprinkle_x, sprinkle_y, sprinkle_z))
    return vertices

vertices = generate_expanded_sprinkles()

# Transparent text texture generation
def create_text_texture(text, font_size=72):
    # Segoe UI Emoji or Segoe UI Symbol ensures the red graphic heart ❤️ renders perfectly
    font_options = ["segoeuiemoji", "segoeuisymbol", "gabriola", "monotypecorsiva"]
    font = pygame.font.SysFont(font_options, font_size, bold=True, italic=True)
    
    # Render base texture in pure white so OpenGL tinting maps perfectly
    text_surface = font.render(text, True, (255, 255, 255)) 
    
    text_data = pygame.image.tostring(text_surface, "RGBA", True)
    width, height = text_surface.get_size()

    texture_id = glGenTextures(1)
    glBindTexture(GL_TEXTURE_2D, texture_id)
    glTexImage2D(GL_TEXTURE_2D, 0, GL_RGBA, width, height, 0, GL_RGBA, GL_UNSIGNED_BYTE, text_data)
    glTexParameteri(GL_TEXTURE_2D, GL_TEXTURE_MIN_FILTER, GL_LINEAR)
    glTexParameteri(GL_TEXTURE_2D, GL_TEXTURE_MAG_FILTER, GL_LINEAR)
    
    return texture_id, width, height

# Initializing with the requested character layout
text_id, txt_w, txt_h = create_text_texture("H❤️A.M", 72)

# OpenGL Setup
gluPerspective(45, (display[0] / display[1]), 0.1, 50.0)
glTranslatef(0.0, 0.0, -25) 
glEnable(GL_BLEND)
glBlendFunc(GL_SRC_ALPHA, GL_ONE_MINUS_SRC_ALPHA)
glEnable(GL_DEPTH_TEST)

# Animation variables
clock = pygame.time.Clock()
angle_y = 0
pulse_time = 0

# Main Loop
running = True
while running:
    for event in pygame.event.get():
        if event.type == pygame.QUIT:
            running = False

    # Update color states
    color_manager.update()
    hc = color_manager.current_heart_color
    tc = color_manager.current_text_color

    # Animation calculations
    pulse_time += 0.04
    pulse_scale = 1.0 + 0.12 * math.sin(pulse_time) 
    angle_y += 0.6  

    # Deep neon ambient dark background
    glClearColor(0.01, 0.0, 0.04, 1.0)
    glClear(GL_COLOR_BUFFER_BIT | GL_DEPTH_BUFFER_BIT)

    glPushMatrix()
    glRotatef(angle_y, 0, 1, 0)      
    glScalef(pulse_scale, pulse_scale, pulse_scale) 
    
    # 1. BACKGROUND SHADOW
    glDisable(GL_DEPTH_TEST)
    glColor4f(hc[0]*0.2, hc[1]*0.2, hc[2]*0.2, 0.2) 
    glPointSize(4.0) 
    glBegin(GL_POINTS)
    for v in vertices:
        glVertex3f(v[0] * 1.05, v[1] * 1.05, v[2] * 1.05)
    glEnd()

    # 2. NEON GLOW LAYER
    glColor4f(hc[0], hc[1], hc[2], 0.25) 
    glPointSize(2.5) 
    glBegin(GL_POINTS)
    for v in vertices:
        glVertex3f(v[0], v[1], v[2])
    glEnd()

    # 3. INNER CORE LAYER
    glEnable(GL_DEPTH_TEST)
    glColor4f(hc[0], hc[1], hc[2], 0.9) 
    glPointSize(1.0) 
    glBegin(GL_POINTS)
    for v in vertices:
        glVertex3f(v[0], v[1], v[2])
    glEnd()

    # 4. CENTERED TEXT
    glEnable(GL_TEXTURE_2D)
    glBindTexture(GL_TEXTURE_2D, text_id)
    glColor4f(tc[0], tc[1], tc[2], 1.0) # Apply dynamic random rainbow neon tinting
    
    tw = txt_w * 0.024
    th = txt_h * 0.024
    
    glBegin(GL_QUADS)
    glTexCoord2f(0, 0); glVertex3f(-tw/2, -th/2, 0.2)  
    glTexCoord2f(1, 0); glVertex3f(tw/2, -th/2, 0.2)
    glTexCoord2f(1, 1); glVertex3f(tw/2, th/2, 0.2)
    glTexCoord2f(0, 1); glVertex3f(-tw/2, th/2, 0.2)
    glEnd()
    glDisable(GL_TEXTURE_2D)

    glPopMatrix()

    pygame.display.flip()
    clock.tick(60)

pygame.quit()

In [12]:
import pygame
from pygame.locals import *
from OpenGL.GL import *
from OpenGL.GLU import *
import math
import random

# Initialize Pygame and OpenGL
pygame.init()
display = (1000, 750) 
pygame.display.set_mode(display, DOUBLEBUF | OPENGL)
pygame.display.set_caption("3D Neon Custom Multi-Color Heart")

# Helper class to handle independent random smooth color shifting
class ColorManager:
    def __init__(self):
        self.current_text_color = [random.random(), random.random(), random.random()]
        self.target_text_color = [random.random(), random.random(), random.random()]
        self.current_heart_color = [random.random(), random.random(), random.random()]
        self.target_heart_color = [random.random(), random.random(), random.random()]
        self.last_change_time = pygame.time.get_ticks()

    def update(self):
        now = pygame.time.get_ticks()
        if now - self.last_change_time > 800: # Color updates every 800ms
            # Generate vibrant, bright target colors
            self.target_text_color = [random.uniform(0.5, 1.0), random.uniform(0.5, 1.0), random.uniform(0.5, 1.0)]
            # Force heart sprinkles to a completely different color pool
            self.target_heart_color = [random.uniform(0.4, 1.0), random.uniform(0.4, 1.0), random.uniform(0.4, 1.0)]
            
            # CRITICAL CHECK: Make sure text color isn't identical to heart sprinkles color
            while math.dist(self.target_text_color, self.target_heart_color) < 0.4:
                self.target_heart_color = [random.uniform(0.4, 1.0), random.uniform(0.4, 1.0), random.uniform(0.4, 1.0)]
                
            self.last_change_time = now

        # Linear interpolation (LERP) for smooth, non-flickering color transitions
        for i in range(3):
            self.current_text_color[i] += (self.target_text_color[i] - self.current_text_color[i]) * 0.08
            self.current_heart_color[i] += (self.target_heart_color[i] - self.current_heart_color[i]) * 0.08

color_manager = ColorManager()

# Generate an expanded 3D Heart with space between sprinkles
def generate_expanded_sprinkles():
    vertices = []
    for u in [i * 0.04 for i in range(-80, 80)]:
        for v in [i * 0.04 for i in range(-40, 40)]:
            base_x = 16 * math.sin(u) ** 3
            base_y = 13 * math.cos(u) - 5 * math.cos(2*u) - 2 * math.cos(3*u) - math.cos(4*u)
            base_z = v * 8  
            
            scale_depth = 1.0 - (abs(v) / 1.5)
            if scale_depth < 0: scale_depth = 0
            base_x *= scale_depth
            base_y *= scale_depth
            
            sprinkle_x = (base_x + random.uniform(-6.0, 6.0)) * 0.24 
            sprinkle_y = (base_y + random.uniform(-6.0, 6.0)) * 0.24
            sprinkle_z = (base_z + random.uniform(-6.0, 6.0)) * 0.24
            
            vertices.append((sprinkle_x, sprinkle_y, sprinkle_z))
    return vertices

vertices = generate_expanded_sprinkles()

# Transparent text texture generation
def create_text_texture(text, font_name="gabriola", font_size=72):
    font = pygame.font.SysFont(font_name, font_size, bold=True, italic=True)
    text_surface = font.render(text, True, (255, 255, 255)) # Pure white base for custom tinting
    text_data = pygame.image.tostring(text_surface, "RGBA", True)
    width, height = text_surface.get_size()

    texture_id = glGenTextures(1)
    glBindTexture(GL_TEXTURE_2D, texture_id)
    glTexImage2D(GL_TEXTURE_2D, 0, GL_RGBA, width, height, 0, GL_RGBA, GL_UNSIGNED_BYTE, text_data)
    glTexParameteri(GL_TEXTURE_2D, GL_TEXTURE_MIN_FILTER, GL_LINEAR)
    glTexParameteri(GL_TEXTURE_2D, GL_TEXTURE_MAG_FILTER, GL_LINEAR)
    
    return texture_id, width, height

# Generate isolated segment textures for independent color maps
h_id, h_w, h_h = create_text_texture("H", "gabriola", 75)
heart_id, heart_w, heart_h = create_text_texture("❤", "segoeuiemoji", 65) # Clean vector representation
am_id, am_w, am_h = create_text_texture("A.M", "gabriola", 75)

# OpenGL Setup
gluPerspective(45, (display[0] / display[1]), 0.1, 50.0)
glTranslatef(0.0, 0.0, -25) 
glEnable(GL_BLEND)
glBlendFunc(GL_SRC_ALPHA, GL_ONE_MINUS_SRC_ALPHA)
glEnable(GL_DEPTH_TEST)

# Animation variables
clock = pygame.time.Clock()
angle_y = 0
pulse_time = 0

# Main Loop
running = True
while running:
    for event in pygame.event.get():
        if event.type == pygame.QUIT:
            running = False

    # Update global color manager
    color_manager.update()
    hc = color_manager.current_heart_color
    tc = color_manager.current_text_color

    # Animation calculations
    pulse_time += 0.04
    pulse_scale = 1.0 + 0.12 * math.sin(pulse_time) 
    angle_y += 0.6  

    # Deep neon ambient dark background
    glClearColor(0.01, 0.0, 0.04, 1.0)
    glClear(GL_COLOR_BUFFER_BIT | GL_DEPTH_BUFFER_BIT)

    glPushMatrix()
    glRotatef(angle_y, 0, 1, 0)      
    glScalef(pulse_scale, pulse_scale, pulse_scale) 
    
    # 1. BACKGROUND SHADOW (Matches the moving sprinkle colors)
    glDisable(GL_DEPTH_TEST)
    glColor4f(hc[0]*0.15, hc[1]*0.15, hc[2]*0.15, 0.2) 
    glPointSize(4.0) 
    glBegin(GL_POINTS)
    for v in vertices:
        glVertex3f(v[0] * 1.05, v[1] * 1.05, v[2] * 1.05)
    glEnd()

    # 2. NEON GLOW LAYER (Outer Sprinkles)
    glColor4f(hc[0], hc[1], hc[2], 0.25) 
    glPointSize(2.5) 
    glBegin(GL_POINTS)
    for v in vertices:
        glVertex3f(v[0], v[1], v[2])
    glEnd()

    # 3. INNER CORE LAYER (Sharp Core Sprinkles)
    glEnable(GL_DEPTH_TEST)
    glColor4f(hc[0], hc[1], hc[2], 0.9) 
    glPointSize(1.0) 
    glBegin(GL_POINTS)
    for v in vertices:
        glVertex3f(v[0], v[1], v[2])
    glEnd()

    # 4. DRAWING SEGMENTED CENTRAL HUD TEXT
    glEnable(GL_TEXTURE_2D)
    
    # --- Part A: Letter "H" (Random Text Color) ---
    glBindTexture(GL_TEXTURE_2D, h_id)
    glColor4f(tc[0], tc[1], tc[2], 1.0)
    hw, hh = h_w * 0.024, h_h * 0.024
    glBegin(GL_QUADS)
    glTexCoord2f(0, 0); glVertex3f(-hw - 0.7, -hh/2, 0.2)  
    glTexCoord2f(1, 0); glVertex3f(-0.7, -hh/2, 0.2)
    glTexCoord2f(1, 1); glVertex3f(-0.7, hh/2, 0.2)
    glTexCoord2f(0, 1); glVertex3f(-hw - 0.7, hh/2, 0.2)
    glEnd()

    # --- Part B: Central Heart "❤" (STRICTLY SOLID NEON RED) ---
    glBindTexture(GL_TEXTURE_2D, heart_id)
    glColor4f(1.0, 0.0, 0.2, 1.0) # Fixed glowing crimson red
    htw, hth = heart_w * 0.024, heart_h * 0.024
    glBegin(GL_QUADS)
    glTexCoord2f(0, 0); glVertex3f(-htw/2, -hth/2, 0.25)  # Positioned exactly in the middle
    glTexCoord2f(1, 0); glVertex3f(htw/2, -hth/2, 0.25)
    glTexCoord2f(1, 1); glVertex3f(htw/2, hth/2, 0.25)
    glTexCoord2f(0, 1); glVertex3f(-htw/2, hth/2, 0.25)
    glEnd()

    # --- Part C: Letters "A.M" (Random Text Color) ---
    glBindTexture(GL_TEXTURE_2D, am_id)
    glColor4f(tc[0], tc[1], tc[2], 1.0)
    amw, amh = am_w * 0.024, am_h * 0.024
    glBegin(GL_QUADS)
    glTexCoord2f(0, 0); glVertex3f(0.7, -amh/2, 0.2)  
    glTexCoord2f(1, 0); glVertex3f(amw + 0.7, -amh/2, 0.2)
    glTexCoord2f(1, 1); glVertex3f(amw + 0.7, amh/2, 0.2)
    glTexCoord2f(0, 1); glVertex3f(0.7, amh/2, 0.2)
    glEnd()

    glDisable(GL_TEXTURE_2D)
    glPopMatrix()

    pygame.display.flip()
    clock.tick(60)

pygame.quit()
